# 07 — Beam Search Decoding

## Objective

Previous Mini-NOVIC experiments used greedy autoregressive decoding.

Greedy decoding selects the single highest-probability next token at every step.

However, the locally best token is not always part of the highest-probability
complete noun sequence.

Beam search instead keeps several candidate sequences during generation and
compares their cumulative sequence probabilities.

This notebook compares:

- Greedy decoding / Beam width 1
- Beam width 3
- Beam width 5

No model retraining is performed.

The same trained decoder and the same controlled image test set are used.

The goals are:

1. Load the trained Mini-NOVIC decoder.
2. Load the same frozen CLIP image encoder.
3. Reproduce greedy generation.
4. Implement beam search.
5. Verify that Beam-1 matches greedy decoding.
6. Compare Beam-1, Beam-3, and Beam-5 image accuracy.
7. Inspect top-k generated noun candidates.
8. Measure Top-1, Top-3, and Top-5 accuracy.
9. Analyze cases where beam search changes the prediction.

In [3]:
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cpu
CUDA available: False


## 1. Environment configuration

A fixed random seed is used for reproducibility.

Beam-search inference itself is deterministic once the trained model and image
embedding are fixed.

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cpu


In [5]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/mini_novic_storage"
)

EMBEDDING_CACHE = (
    DRIVE_ROOT
    / "embeddings"
    / "text_embedding_dataset"
    / "clip_text_embeddings.pt"
)

CHECKPOINT_DIR = (
    DRIVE_ROOT
    / "checkpoints"
    / "noise_ablation"
)

IMAGE_TEST_DIR = (
    DRIVE_ROOT
    / "datasets"
    / "image_transfer_test"
)

RESULT_DIR = (
    DRIVE_ROOT
    / "results"
    / "beam_search"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Embedding cache :", EMBEDDING_CACHE)
print("Checkpoint dir  :", CHECKPOINT_DIR)
print("Image test dir  :", IMAGE_TEST_DIR)
print("Result dir      :", RESULT_DIR)

Embedding cache : /content/drive/MyDrive/mini_novic_storage/embeddings/text_embedding_dataset/clip_text_embeddings.pt
Checkpoint dir  : /content/drive/MyDrive/mini_novic_storage/checkpoints/noise_ablation
Image test dir  : /content/drive/MyDrive/mini_novic_storage/datasets/image_transfer_test
Result dir      : /content/drive/MyDrive/mini_novic_storage/results/beam_search


In [7]:
for path in CHECKPOINT_DIR.glob("*.pt"):
    print(path.name)

A1_gaussian_best.pt
A2_novic_style_best.pt
sweep_gaussian_0p000_best.pt
sweep_gaussian_0p010_best.pt
sweep_gaussian_0p030_best.pt
sweep_gaussian_0p050_best.pt
sweep_gaussian_0p075_best.pt
sweep_gaussian_0p100_best.pt
sweep_gaussian_0p150_best.pt
A3_fair_mixed_sigma_0p05_best.pt


In [8]:
BEAM_MODEL_PATH = (
    CHECKPOINT_DIR
    / "sweep_gaussian_0p050_best.pt"
)

print("Using checkpoint:")
print(BEAM_MODEL_PATH)

print(
    "Exists:",
    BEAM_MODEL_PATH.exists()
)

Using checkpoint:
/content/drive/MyDrive/mini_novic_storage/checkpoints/noise_ablation/sweep_gaussian_0p050_best.pt
Exists: True


## 2. Load the original CLIP configuration

The same CLIP model used to generate the decoder's training text embeddings must
also be used for image inference.

Using embeddings from a different CLIP model would make the experiment invalid.

In [9]:
cache = torch.load(
    EMBEDDING_CACHE,
    map_location="cpu"
)

MODEL_NAME = cache["model_name"]
PRETRAINED = cache["pretrained"]

print("CLIP model:", MODEL_NAME)
print("Pretrained:", PRETRAINED)

CLIP model: ViT-B-32
Pretrained: laion2b_s34b_b79k


# Install and load OpenCLIP

In [10]:
!pip install -q open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00


In [11]:
import open_clip

In [12]:
clip_model, _, preprocess = (
    open_clip.create_model_and_transforms(
        MODEL_NAME,
        pretrained=PRETRAINED
    )
)

clip_model = (
    clip_model
    .to(device)
    .eval()
)

for parameter in clip_model.parameters():
    parameter.requires_grad = False

print("CLIP loaded successfully")

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

CLIP loaded successfully


In [13]:
trainable_clip_parameters = sum(
    p.numel()
    for p in clip_model.parameters()
    if p.requires_grad
)

print(
    "Trainable CLIP parameters:",
    trainable_clip_parameters
)


Trainable CLIP parameters: 0


##  Load decoder checkpoint metadata

In [14]:
checkpoint = torch.load(
    BEAM_MODEL_PATH,
    map_location=device
)

print(
    "Experiment:",
    checkpoint.get(
        "experiment",
        "not stored"
    )
)

print(
    "Best epoch:",
    checkpoint["epoch"]
)

print(
    "Validation loss:",
    checkpoint["val_loss"]
)

Experiment: sweep_gaussian_0p050
Best epoch: 30
Validation loss: 0.0036839190796017648


## Load architecture configuration

In [15]:
CLEAN_CHECKPOINT_PATH = (
    DRIVE_ROOT
    / "checkpoints"
    / "full_decoder_training"
    / "best_full_decoder.pt"
)

clean_checkpoint = torch.load(
    CLEAN_CHECKPOINT_PATH,
    map_location="cpu"
)

CLIP_DIM = clean_checkpoint["clip_dim"]
HIDDEN_DIM = clean_checkpoint["hidden_dim"]
PREFIX_LEN = clean_checkpoint["prefix_len"]

NUM_LAYERS = clean_checkpoint["num_layers"]
NUM_HEADS = clean_checkpoint["num_heads"]

MAX_SEQ_LEN = clean_checkpoint["max_seq_len"]

vocab = clean_checkpoint["vocab"]
token_to_id = clean_checkpoint["token_to_id"]
id_to_token = clean_checkpoint["id_to_token"]

PAD_ID = clean_checkpoint["pad_id"]
EOS_ID = clean_checkpoint["eos_id"]

VOCAB_SIZE = len(vocab)

print("CLIP dimension :", CLIP_DIM)
print("Hidden size    :", HIDDEN_DIM)
print("Prefix length  :", PREFIX_LEN)
print("Layers         :", NUM_LAYERS)
print("Heads          :", NUM_HEADS)
print("Vocabulary size:", VOCAB_SIZE)
print("Max seq length :", MAX_SEQ_LEN)

CLIP dimension : 512
Hidden size    : 256
Prefix length  : 4
Layers         : 4
Heads          : 4
Vocabulary size: 58
Max seq length : 4


## 3. Recreate the trained decoder

The decoder architecture must exactly match the architecture used during
training.

Prefix tokens have unrestricted attention to other prefix tokens.

Generated noun tokens use causal attention.

In [16]:
def build_attention_mask(
    prefix_len,
    token_len,
    device
):

    total_len = (
        prefix_len
        + token_len
    )

    mask = torch.full(
        (total_len, total_len),
        float("-inf"),
        device=device
    )

    # Prefix-to-prefix full attention
    mask[
        :prefix_len,
        :prefix_len
    ] = 0

    # Causal noun-token attention
    for i in range(token_len):

        row = prefix_len + i

        mask[
            row,
            :prefix_len + i + 1
        ] = 0

    return mask

# Decoder class

In [17]:
class MiniNOVICDecoder(nn.Module):

    def __init__(
        self,
        clip_dim,
        vocab_size,
        hidden_dim=256,
        prefix_len=4,
        max_seq_len=4,
        num_layers=4,
        num_heads=4,
        dropout=0.1,
    ):

        super().__init__()

        self.hidden_dim = hidden_dim
        self.prefix_len = prefix_len
        self.max_seq_len = max_seq_len

        # CLIP embedding -> prefix tokens
        self.prefix_mapper = nn.Linear(
            clip_dim,
            prefix_len * hidden_dim
        )

        # Noun token embeddings
        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_dim
        )

        # Learned positional embeddings
        self.position_embedding = nn.Embedding(
            prefix_len + max_seq_len,
            hidden_dim
        )

        transformer_layer = (
            nn.TransformerEncoderLayer(
                d_model=hidden_dim,
                nhead=num_heads,
                dim_feedforward=hidden_dim * 4,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
        )

        self.transformer = nn.TransformerEncoder(
            transformer_layer,
            num_layers=num_layers
        )

        self.output_projection = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(
        self,
        clip_embeddings,
        token_ids
    ):

        batch_size = (
            clip_embeddings.size(0)
        )

        # -----------------------------
        # CLIP embedding -> prefixes
        # -----------------------------

        prefix = self.prefix_mapper(
            clip_embeddings
        )

        prefix = prefix.view(
            batch_size,
            self.prefix_len,
            self.hidden_dim
        )

        # -----------------------------
        # Generated tokens -> vectors
        # -----------------------------

        token_vectors = self.token_embedding(
            token_ids
        )

        # -----------------------------
        # Concatenate
        # -----------------------------

        x = torch.cat(
            [
                prefix,
                token_vectors
            ],
            dim=1
        )

        # -----------------------------
        # Position embeddings
        # -----------------------------

        total_len = x.size(1)

        positions = torch.arange(
            total_len,
            device=x.device
        )

        x = (
            x
            + self.position_embedding(
                positions
            ).unsqueeze(0)
        )

        # -----------------------------
        # Attention mask
        # -----------------------------

        mask = build_attention_mask(
            self.prefix_len,
            token_ids.size(1),
            x.device
        )

        # -----------------------------
        # Transformer
        # -----------------------------

        x = self.transformer(
            x,
            mask=mask
        )

        # -----------------------------
        # Shifted prediction states
        #
        # P4 -> token 1
        # T1 -> token 2
        # ...
        # -----------------------------

        prediction_states = x[
            :,
            self.prefix_len - 1:,
            :
        ]

        logits = self.output_projection(
            prediction_states
        )

        return logits

# Create and load decoder

In [18]:
beam_model = MiniNOVICDecoder(
    clip_dim=CLIP_DIM,
    vocab_size=VOCAB_SIZE,
    hidden_dim=HIDDEN_DIM,
    prefix_len=PREFIX_LEN,
    max_seq_len=MAX_SEQ_LEN,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    dropout=0.1,
).to(device)

/tmp/ipykernel_614/2778019281.py:51: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


In [19]:
beam_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

beam_model.eval()

print(
    "Decoder loaded successfully"
)

Decoder loaded successfully


# Decode tokens

In [20]:
def decode_tokens(token_ids):

    words = []

    for token_id in token_ids:

        if token_id == EOS_ID:
            break

        if token_id == PAD_ID:
            continue

        words.append(
            id_to_token[token_id]
        )

    return " ".join(words)

In [21]:
print(
    decode_tokens(
        [
            token_to_id["fire"],
            token_to_id["truck"],
            EOS_ID
        ]
    )
)

fire truck


## 4. Load the controlled image dataset

The corrected image test set from Notebook 06 is reused.

No image labels are used for training.

The labels are used only for evaluation.

In [22]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
}

records = []

for noun_dir in IMAGE_TEST_DIR.iterdir():

    if not noun_dir.is_dir():
        continue

    for image_path in noun_dir.iterdir():

        if (
            image_path.suffix.lower()
            in IMAGE_EXTENSIONS
        ):

            records.append({
                "image_path":
                    str(image_path),

                "expected":
                    noun_dir.name
            })

image_df = pd.DataFrame(
    records
)

print(
    "Number of images:",
    len(image_df)
)

print(
    "Number of classes:",
    image_df[
        "expected"
    ].nunique()
)

display(
    image_df.head(10)
)

Number of images: 38
Number of classes: 10


,image_path,expected
0,/content/drive/MyDrive/mini_novic_storage/data...,cat
1,/content/drive/MyDrive/mini_novic_storage/data...,cat
2,/content/drive/MyDrive/mini_novic_storage/data...,cat
3,/content/drive/MyDrive/mini_novic_storage/data...,cat
4,/content/drive/MyDrive/mini_novic_storage/data...,dog
5,/content/drive/MyDrive/mini_novic_storage/data...,dog
6,/content/drive/MyDrive/mini_novic_storage/data...,dog
7,/content/drive/MyDrive/mini_novic_storage/data...,dog
8,/content/drive/MyDrive/mini_novic_storage/data...,fire truck
9,/content/drive/MyDrive/mini_novic_storage/data...,fire truck


# Image encoder

In [23]:
@torch.no_grad()
def encode_image(
    image_path
):

    image = Image.open(
        image_path
    ).convert("RGB")

    image_tensor = (
        preprocess(image)
        .unsqueeze(0)
        .to(device)
    )

    embedding = (
        clip_model.encode_image(
            image_tensor
        )
    )

    embedding = F.normalize(
        embedding,
        dim=-1
    )

    return embedding

# Test one image embedding

In [24]:
sample_path = (
    image_df.iloc[0][
        "image_path"
    ]
)

sample_embedding = encode_image(
    sample_path
)

print(
    "Embedding shape:",
    sample_embedding.shape
)

print(
    "Embedding norm:",
    sample_embedding
    .norm(dim=-1)
    .item()
)

Embedding shape: torch.Size([1, 512])
Embedding norm: 1.0


## 5. Greedy decoding baseline

Greedy decoding selects the highest-probability token at each autoregressive
generation step.

Only one sequence is retained.

In [25]:
@torch.no_grad()
def generate_greedy(
    model,
    clip_embedding,
    max_length=MAX_SEQ_LEN
):

    model.eval()

    if clip_embedding.dim() == 1:

        clip_embedding = (
            clip_embedding.unsqueeze(0)
        )

    clip_embedding = (
        clip_embedding.to(device)
    )

    generated = []

    for _ in range(max_length):

        if len(generated) == 0:

            token_inputs = torch.empty(
                (1, 0),
                dtype=torch.long,
                device=device
            )

        else:

            token_inputs = torch.tensor(
                [generated],
                dtype=torch.long,
                device=device
            )

        logits = model(
            clip_embedding,
            token_inputs
        )

        next_token = (
            logits[
                0,
                -1,
                :
            ]
            .argmax()
            .item()
        )

        if next_token in [
            EOS_ID,
            PAD_ID
        ]:
            break

        generated.append(
            next_token
        )

    return generated

## Test greedy on one image

In [26]:
sample_row = image_df.iloc[0]

embedding = encode_image(
    sample_row["image_path"]
)

greedy_ids = generate_greedy(
    beam_model,
    embedding
)

greedy_prediction = decode_tokens(
    greedy_ids
)

print(
    "Expected :",
    sample_row["expected"]
)

print(
    "Greedy   :",
    greedy_prediction
)

Expected : cat
Greedy   : cat


## 6. Beam search

Greedy decoding keeps only one sequence.

For example:

Step 1:

fire       logP = -0.8
truck      logP = -1.0
bus        logP = -1.4

Greedy keeps only:

fire

Beam search with width 3 keeps:

fire
truck
bus

Each sequence is then expanded again.

A beam stores:

(tokens, cumulative log-probability, finished)

The score of a sequence is:

log P(t1, t2, ..., tn)
=
log P(t1)
+ log P(t2 | t1)
+ ...
+ log P(tn | t1,...,tn-1)

Log-probabilities are used because multiplying many small probabilities is
numerically unstable.

In [27]:
@torch.no_grad()
def generate_beam_search(
    model,
    clip_embedding,
    beam_width=3,
    max_length=MAX_SEQ_LEN,
    return_top_k=None
):

    model.eval()

    if clip_embedding.dim() == 1:

        clip_embedding = (
            clip_embedding.unsqueeze(0)
        )

    clip_embedding = (
        clip_embedding.to(device)
    )

    # Beam structure:
    #
    # (
    #   generated_tokens,
    #   cumulative_log_probability,
    #   finished
    # )

    beams = [
        (
            [],
            0.0,
            False
        )
    ]

    for _ in range(max_length):

        candidates = []

        for (
            tokens,
            score,
            finished
        ) in beams:

            # -------------------------
            # Finished beam
            # -------------------------

            if finished:

                candidates.append(
                    (
                        tokens,
                        score,
                        True
                    )
                )

                continue

            # -------------------------
            # Current token input
            # -------------------------

            if len(tokens) == 0:

                token_inputs = torch.empty(
                    (1, 0),
                    dtype=torch.long,
                    device=device
                )

            else:

                token_inputs = torch.tensor(
                    [tokens],
                    dtype=torch.long,
                    device=device
                )

            # -------------------------
            # Predict next token
            # -------------------------

            logits = model(
                clip_embedding,
                token_inputs
            )

            next_logits = logits[
                0,
                -1,
                :
            ]

            log_probs = F.log_softmax(
                next_logits,
                dim=-1
            )

            # PAD should not be emitted
            log_probs[
                PAD_ID
            ] = float("-inf")

            # -------------------------
            # Best next tokens
            # -------------------------

            top_log_probs, top_ids = (
                torch.topk(
                    log_probs,
                    k=min(
                        beam_width,
                        log_probs.size(0)
                    )
                )
            )

            # -------------------------
            # Expand beam
            # -------------------------

            for (
                token_log_prob,
                token_id
            ) in zip(
                top_log_probs.tolist(),
                top_ids.tolist()
            ):

                new_score = (
                    score
                    + token_log_prob
                )

                if token_id == EOS_ID:

                    candidates.append(
                        (
                            tokens.copy(),
                            new_score,
                            True
                        )
                    )

                else:

                    candidates.append(
                        (
                            tokens
                            + [token_id],

                            new_score,

                            False
                        )
                    )

        # -------------------------
        # Keep best beams
        # -------------------------

        candidates.sort(
            key=lambda x: x[1],
            reverse=True
        )

        beams = candidates[
            :beam_width
        ]

        # -------------------------
        # Stop if all completed
        # -------------------------

        if all(
            finished
            for _, _, finished in beams
        ):
            break

    beams.sort(
        key=lambda x: x[1],
        reverse=True
    )

    if return_top_k is None:
        return_top_k = beam_width

    return beams[
        :return_top_k
    ]

## Test beam search on one image

In [28]:
beam_candidates = generate_beam_search(
    beam_model,
    embedding,
    beam_width=5,
    return_top_k=5
)

print(
    "Expected:",
    sample_row["expected"]
)

print()

for rank, (
    tokens,
    score,
    finished
) in enumerate(
    beam_candidates,
    start=1
):

    print(
        f"{rank}. "
        f"{decode_tokens(tokens):20s} "
        f"logP={score:.4f} "
        f"finished={finished}"
    )

Expected: cat

1. cat                  logP=-0.0037 finished=True
2. car                  logP=-6.1630 finished=True
3. giraffe              logP=-6.9398 finished=True
4. bird                 logP=-8.1193 finished=True
5. bottle               logP=-8.6471 finished=True


# Critical sanity test: Beam-1 = Greedy

In [29]:
mismatches = []

for i in range(
    len(image_df)
):

    row = image_df.iloc[i]

    embedding = encode_image(
        row["image_path"]
    )

    greedy_ids = generate_greedy(
        beam_model,
        embedding
    )

    beam1 = generate_beam_search(
        beam_model,
        embedding,
        beam_width=1,
        return_top_k=1
    )

    beam1_ids = beam1[0][0]

    greedy_name = decode_tokens(
        greedy_ids
    )

    beam1_name = decode_tokens(
        beam1_ids
    )

    if greedy_name != beam1_name:

        mismatches.append({
            "image_path":
                row["image_path"],

            "greedy":
                greedy_name,

            "beam_1":
                beam1_name
        })

In [30]:
print(
    "Greedy vs Beam-1 mismatches:",
    len(mismatches)
)

if len(mismatches) > 0:

    display(
        pd.DataFrame(
            mismatches
        )
    )

Greedy vs Beam-1 mismatches: 0


# Image evaluation function

In [31]:
@torch.no_grad()
def evaluate_beam_images(
    model,
    image_df,
    beam_width
):

    predictions = []
    scores = []

    for _, row in image_df.iterrows():

        embedding = encode_image(
            row["image_path"]
        )

        beams = generate_beam_search(
            model,
            embedding,
            beam_width=beam_width,
            return_top_k=1
        )

        tokens, score, finished = (
            beams[0]
        )

        prediction = decode_tokens(
            tokens
        )

        predictions.append(
            prediction
        )

        scores.append(
            score
        )

    results = image_df.copy()

    results[
        "generated"
    ] = predictions

    results[
        "log_probability"
    ] = scores

    results[
        "correct"
    ] = (
        results["expected"]
        ==
        results["generated"]
    )

    accuracy = (
        results["correct"]
        .mean()
    )

    return accuracy, results

## 7. Beam-width comparison

The same images are evaluated using three search widths:

- Beam-1
- Beam-3
- Beam-5

Only inference-time search changes.

In [32]:
BEAM_WIDTHS = [
    1,
    3,
    5
]

beam_results = {}
summary_records = []

for beam_width in BEAM_WIDTHS:

    accuracy, results = (
        evaluate_beam_images(
            beam_model,
            image_df,
            beam_width
        )
    )

    beam_results[
        beam_width
    ] = results

    summary_records.append({
        "beam_width":
            beam_width,

        "accuracy":
            accuracy,

        "accuracy_percent":
            accuracy * 100
    })

    print(
        f"Beam-{beam_width}: "
        f"{accuracy*100:.2f}%"
    )

Beam-1: 89.47%
Beam-3: 89.47%
Beam-5: 89.47%


# Summary table

In [33]:
beam_summary = pd.DataFrame(
    summary_records
)

display(
    beam_summary
)

,beam_width,accuracy,accuracy_percent
0,1,0.894737,89.473684
1,3,0.894737,89.473684
2,5,0.894737,89.473684


In [34]:
beam_summary.to_csv(
    RESULT_DIR
    / "beam_search_summary.csv",
    index=False
)

# Compare predictions

In [35]:
beam_comparison = pd.DataFrame({
    "image_path":
        image_df["image_path"].values,

    "expected":
        image_df["expected"].values,

    "beam_1":
        beam_results[1][
            "generated"
        ].values,

    "beam_3":
        beam_results[3][
            "generated"
        ].values,

    "beam_5":
        beam_results[5][
            "generated"
        ].values,
})

display(
    beam_comparison
)

,image_path,expected,beam_1,beam_3,beam_5
0,/content/drive/MyDrive/mini_novic_storage/data...,cat,cat,cat,cat
1,/content/drive/MyDrive/mini_novic_storage/data...,cat,cat,cat,cat
2,/content/drive/MyDrive/mini_novic_storage/data...,cat,cat,cat,cat
3,/content/drive/MyDrive/mini_novic_storage/data...,cat,cat,cat,cat
4,/content/drive/MyDrive/mini_novic_storage/data...,dog,dog,dog,dog
5,/content/drive/MyDrive/mini_novic_storage/data...,dog,dog,dog,dog
6,/content/drive/MyDrive/mini_novic_storage/data...,dog,dog,dog,dog
7,/content/drive/MyDrive/mini_novic_storage/data...,dog,dog,dog,dog
8,/content/drive/MyDrive/mini_novic_storage/data...,fire truck,fire truck,fire truck,fire truck
9,/content/drive/MyDrive/mini_novic_storage/data...,fire truck,fire truck,fire truck,fire truck


In [36]:
changed_predictions = (
    beam_comparison[
        (
            beam_comparison[
                "beam_1"
            ]
            !=
            beam_comparison[
                "beam_3"
            ]
        )
        |
        (
            beam_comparison[
                "beam_1"
            ]
            !=
            beam_comparison[
                "beam_5"
            ]
        )
    ]
)

print(
    "Predictions changed by beam search:",
    len(changed_predictions)
)

display(
    changed_predictions
)

Predictions changed by beam search: 0


,image_path,expected,beam_1,beam_3,beam_5


In [37]:
beam_comparison[
    "beam1_correct"
] = (
    beam_comparison[
        "expected"
    ]
    ==
    beam_comparison[
        "beam_1"
    ]
)

beam_comparison[
    "beam3_correct"
] = (
    beam_comparison[
        "expected"
    ]
    ==
    beam_comparison[
        "beam_3"
    ]
)

beam_comparison[
    "beam5_correct"
] = (
    beam_comparison[
        "expected"
    ]
    ==
    beam_comparison[
        "beam_5"
    ]
)

In [38]:
beam3_improved = beam_comparison[
    (~beam_comparison["beam1_correct"])
    &
    beam_comparison["beam3_correct"]
]

beam3_regressed = beam_comparison[
    beam_comparison["beam1_correct"]
    &
    (~beam_comparison["beam3_correct"])
]

beam5_improved = beam_comparison[
    (~beam_comparison["beam1_correct"])
    &
    beam_comparison["beam5_correct"]
]

beam5_regressed = beam_comparison[
    beam_comparison["beam1_correct"]
    &
    (~beam_comparison["beam5_correct"])
]

print(
    "Beam-3 improvements:",
    len(beam3_improved)
)

print(
    "Beam-3 regressions:",
    len(beam3_regressed)
)

print(
    "Beam-5 improvements:",
    len(beam5_improved)
)

print(
    "Beam-5 regressions:",
    len(beam5_regressed)
)

Beam-3 improvements: 0
Beam-3 regressions: 0
Beam-5 improvements: 0
Beam-5 regressions: 0


In [39]:
# ============================================================
# COMPACT BEAM-SEARCH EVALUATION
# ============================================================

BEAM_WIDTHS = [1, 3, 5]

# ------------------------------------------------------------
# 1. Evaluate top-1 prediction for Beam-1, Beam-3, Beam-5
# ------------------------------------------------------------

beam_results = {}
summary = []

for width in BEAM_WIDTHS:

    predictions = []

    for _, row in image_df.iterrows():

        embedding = encode_image(
            row["image_path"]
        )

        beams = generate_beam_search(
            beam_model,
            embedding,
            beam_width=width,
            return_top_k=1
        )

        tokens, score, finished = beams[0]

        predictions.append(
            decode_tokens(tokens)
        )

    results = image_df.copy()
    results["generated"] = predictions
    results["correct"] = (
        results["expected"]
        ==
        results["generated"]
    )

    accuracy = results["correct"].mean()

    beam_results[width] = results

    summary.append({
        "beam_width": width,
        "accuracy_percent": accuracy * 100
    })

    print(
        f"Beam-{width} accuracy: "
        f"{accuracy*100:.2f}%"
    )


# ------------------------------------------------------------
# 2. Compare Beam-1 / Beam-3 / Beam-5 directly
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "image_path":
        image_df["image_path"].values,

    "expected":
        image_df["expected"].values,

    "beam_1":
        beam_results[1]["generated"].values,

    "beam_3":
        beam_results[3]["generated"].values,

    "beam_5":
        beam_results[5]["generated"].values,
})

comparison["beam1_correct"] = (
    comparison["expected"]
    == comparison["beam_1"]
)

comparison["beam3_correct"] = (
    comparison["expected"]
    == comparison["beam_3"]
)

comparison["beam5_correct"] = (
    comparison["expected"]
    == comparison["beam_5"]
)

changed = comparison[
    (comparison["beam_1"] != comparison["beam_3"])
    |
    (comparison["beam_1"] != comparison["beam_5"])
]

beam3_improved = (
    (~comparison["beam1_correct"])
    &
    comparison["beam3_correct"]
).sum()

beam3_regressed = (
    comparison["beam1_correct"]
    &
    (~comparison["beam3_correct"])
).sum()

beam5_improved = (
    (~comparison["beam1_correct"])
    &
    comparison["beam5_correct"]
).sum()

beam5_regressed = (
    comparison["beam1_correct"]
    &
    (~comparison["beam5_correct"])
).sum()

print("\nChanged predictions:", len(changed))
print(
    "Beam-3 improvements / regressions:",
    beam3_improved,
    "/",
    beam3_regressed
)
print(
    "Beam-5 improvements / regressions:",
    beam5_improved,
    "/",
    beam5_regressed
)

if len(changed) > 0:
    display(changed)


# ------------------------------------------------------------
# 3. Generate Top-5 candidates using Beam-5
# ------------------------------------------------------------

top5_records = []

for _, row in image_df.iterrows():

    embedding = encode_image(
        row["image_path"]
    )

    beams = generate_beam_search(
        beam_model,
        embedding,
        beam_width=5,
        return_top_k=5
    )

    record = {
        "image_path": row["image_path"],
        "expected": row["expected"]
    }

    for rank, (tokens, score, finished) in enumerate(
        beams,
        start=1
    ):
        record[f"rank_{rank}"] = decode_tokens(tokens)
        record[f"score_{rank}"] = score

    top5_records.append(record)

top5_df = pd.DataFrame(top5_records)


# ------------------------------------------------------------
# 4. Top-1 / Top-3 / Top-5 accuracy
# ------------------------------------------------------------

def topk_accuracy(df, k):

    correct = []

    for _, row in df.iterrows():

        predictions = [
            row[f"rank_{i}"]
            for i in range(1, k + 1)
        ]

        correct.append(
            row["expected"] in predictions
        )

    return np.mean(correct)


print("\nTOP-K RESULTS")

for k in [1, 3, 5]:

    acc = topk_accuracy(
        top5_df,
        k
    )

    print(
        f"Top-{k} accuracy: "
        f"{acc*100:.2f}%"
    )


# ------------------------------------------------------------
# 5. Cases where Top-1 was wrong but correct noun is in Top-5
# ------------------------------------------------------------

interesting = []

for _, row in top5_df.iterrows():

    expected = row["expected"]

    top5 = [
        row[f"rank_{i}"]
        for i in range(1, 6)
    ]

    if (
        top5[0] != expected
        and expected in top5
    ):

        interesting.append({
            "image_path":
                row["image_path"],

            "expected":
                expected,

            "top1":
                top5[0],

            "correct_rank":
                top5.index(expected) + 1,

            "top5":
                top5
        })

interesting_df = pd.DataFrame(
    interesting
)

print(
    "\nWrong Top-1 but correct noun in Top-5:",
    len(interesting_df)
)

if len(interesting_df) > 0:
    display(interesting_df)


# ------------------------------------------------------------
# 6. Save important outputs
# ------------------------------------------------------------

pd.DataFrame(summary).to_csv(
    RESULT_DIR / "beam_search_summary.csv",
    index=False
)

comparison.to_csv(
    RESULT_DIR / "beam_prediction_comparison.csv",
    index=False
)

top5_df.to_csv(
    RESULT_DIR / "beam_top5_predictions.csv",
    index=False
)

print("\nSaved beam-search results.")

Beam-1 accuracy: 89.47%
Beam-3 accuracy: 89.47%
Beam-5 accuracy: 89.47%

Changed predictions: 0
Beam-3 improvements / regressions: 0 / 0
Beam-5 improvements / regressions: 0 / 0

TOP-K RESULTS
Top-1 accuracy: 89.47%
Top-3 accuracy: 100.00%
Top-5 accuracy: 100.00%

Wrong Top-1 but correct noun in Top-5: 4


,image_path,expected,top1,correct_rank,top5
0,/content/drive/MyDrive/mini_novic_storage/data...,fire truck,truck,2,"[truck, fire truck, car, monitor, truck truck]"
1,/content/drive/MyDrive/mini_novic_storage/data...,chair,office chair,2,"[office chair, chair, bicycle, keyboard, dinin..."
2,/content/drive/MyDrive/mini_novic_storage/data...,chair,dining table,2,"[dining table, chair, office chair, table, zebra]"
3,/content/drive/MyDrive/mini_novic_storage/data...,bus,truck,3,"[truck, fire truck, bus, street sign, train]"



Saved beam-search results.


# Conclusion

Beam-search decoding was evaluated using beam widths 1, 3, and 5.

All three beam widths achieved the same Top-1 exact-match accuracy of 89.47%.
No image changed its highest-ranked prediction when the beam width was increased.

However, Top-k evaluation revealed additional useful information:

- Top-1 accuracy: 89.47%
- Top-3 accuracy: 100%
- Top-5 accuracy: 100%

For four incorrectly ranked Top-1 predictions, the correct noun was still present
among the highest-probability beam candidates.

Examples included:

- "fire truck" ranked second behind "truck"
- "chair" ranked second behind "office chair"
- "chair" ranked second behind "dining table"
- "bus" ranked third behind other vehicle predictions

Therefore, beam search did not improve Top-1 classification accuracy, because
the decoder itself assigned higher sequence probability to another noun.
Increasing the beam width cannot change this ranking.

Nevertheless, the 100% Top-3 accuracy indicates that the decoder retained the
correct semantic concept among its high-probability predictions for every test
image.

The limited Top-1 improvement is likely related to the small controlled dataset,
the small training vocabulary, and the short one- or two-token noun sequences,
for which greedy decoding is already a strong approximation.